# Fase 2 — Alinhamento Multimodal com Distilação e Representação

**Notebook:** `02_multimodal_alignment.ipynb`  
**Fase:** 2 de 4  
**Projeto:** Arquitetura Híbrida Multimodal em torno de bitnet.cpp  

---

## Resumo

Este notebook implementa a **Fase 2** do pipeline: alinhamento multimodal com descongelamento seletivo dos blocos superiores do backbone BitNet, introdução de *distillation loss* (KL divergence entre logits do teacher e do student) e *representation alignment loss* (distância entre embeddings intermediários do encoder e do conector), conforme Seção 5.3 da especificação técnica.

O encoder permanece em FP16/BF16 e o conector é carregado a partir do checkpoint da Fase 1. O backbone é parcialmente descongelado nas suas camadas superiores, permitindo adaptação ao domínio multimodal sem destruir as representações aprendidas nas camadas inferiores.

---

## Índice

1. [Instalação de Dependências](#1)
2. [Configuração Global](#2)
3. [Montagem do Google Drive](#3)
4. [Fundamentação Teórica](#4)
5. [Carregamento dos Componentes](#5)
6. [Descongelamento Seletivo do Backbone](#6)
7. [Definição da Loss Composta](#7)
8. [Preparação dos Dados](#8)
9. [Loop de Alinhamento Multimodal](#9)
10. [Métricas de Alinhamento](#10)
11. [Persistência dos Artefatos](#11)
12. [Conclusões e Próximos Passos](#12)

In [ ]:
!pip install -q transformers==4.44.0 accelerate==0.33.0 datasets==2.21.0 \
    sentencepiece==0.2.0 safetensors==0.4.3 einops==0.8.0 timm==1.0.9
print("Instalação concluída.")

In [ ]:
import json, logging, math, random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s — %(message)s",
)
logger = logging.getLogger("phase2")

# Reprodutibilidade
SEED: int = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Constantes da fase
TEACHER_MODEL_ID: str   = "microsoft/bitnet-b1.58-2B-4T"
VISION_ENCODER_ID: str  = "openai/clip-vit-large-patch14"
DTYPE_HIGH: torch.dtype = torch.bfloat16
N_UNFREEZE_LAYERS: int  = 4       # Número de blocos superiores a descongelar
LR_BACKBONE: float      = 5e-5    # LR menor para backbone (evitar catastrophic forgetting)
LR_CONNECTOR: float     = 2e-4
N_EPOCHS: int           = 3
BATCH_SIZE: int         = 2
GRAD_CLIP: float        = 1.0
WARMUP_STEPS: int       = 150
LOG_INTERVAL: int       = 50
W_LANGUAGE: float       = 1.0
W_DISTILL: float        = 0.5     # Peso da KL distillation loss
W_ALIGN: float          = 0.1     # Peso da representation alignment loss
TEMP_DISTILL: float     = 4.0     # Temperatura da distilação
DRIVE_PROJECT_DIR: str  = "/content/drive/MyDrive/multimodal-ternary-llm"
PHASE_NAME: str         = "phase2_alignment"

logger.info("Configuração da Fase 2 inicializada.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    logger.info("Google Drive montado.")
except ImportError:
    logger.warning("Ambiente não-Colab. Saltando montagem do Drive.")

CHECKPOINT_DIR = Path(DRIVE_PROJECT_DIR) / "checkpoints" / PHASE_NAME
PHASE1_CKPT    = Path(DRIVE_PROJECT_DIR) / "checkpoints" / "phase1_connector" / "connector_phase1.pt"
METRICS_DIR    = Path(DRIVE_PROJECT_DIR) / "metrics"

for d in (CHECKPOINT_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Carregar d_model da Fase 0
p0 = METRICS_DIR / "phase0_baseline_metrics.json"
D_MODEL: int = json.load(open(p0))["d_model"] if p0.exists() else 2048
D_ENC_VISION: int = 1024
CONNECTOR_HIDDEN: int = max(D_ENC_VISION, D_MODEL)
logger.info("d_model=%d", D_MODEL)

## 4. Fundamentação Teórica

### 4.1 Distillation-Aware Training

O *distillation-aware training* empregado nesta fase segue o protocolo do BitVLA: os logits do modelo *teacher* FP16/BF16 são utilizados como sinal de supervisão suave, via KL divergence, para guiar o modelo *student* (backbone parcialmente descongelado + conector) na preservação da qualidade de geração durante a adaptação multimodal.

A *temperature* de distilação controla a suavidade das distribuições de probabilidade: valores maiores produzem distribuições mais uniformes, facilitando a transferência de conhecimento sobre a incerteza do teacher.

### 4.2 Representation Alignment Loss

A *representation alignment loss* minimiza a distância entre as representações intermediárias do encoder (espaço visual) e as saídas do conector (espaço do backbone). Utiliza-se similaridade de cosseno negada como métrica, seguindo a prática do BitVLA.

### 4.3 Descongelamento Seletivo

Apenas os `N_UNFREEZE_LAYERS` blocos superiores do backbone são descongelados. As camadas inferiores, que codificam representações linguísticas gerais, permanecem congeladas para evitar *catastrophic forgetting*.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, CLIPVisionModel, CLIPProcessor

# Teacher (congelado completamente)
logger.info("Carregando teacher (congelado): %s", TEACHER_MODEL_ID)
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=DTYPE_HIGH, device_map="auto", trust_remote_code=True
)
for p in teacher.parameters(): p.requires_grad = False
teacher.eval()

# Student backbone (carregado separadamente — será parcialmente descongelado)
logger.info("Carregando student backbone: %s", TEACHER_MODEL_ID)
student_backbone = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=DTYPE_HIGH, device_map="auto", trust_remote_code=True
)
# Congelar todo o backbone inicialmente
for p in student_backbone.parameters(): p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Vision encoder (congelado)
vision_encoder = CLIPVisionModel.from_pretrained(VISION_ENCODER_ID, torch_dtype=DTYPE_HIGH).to(DEVICE)
vision_processor = CLIPProcessor.from_pretrained(VISION_ENCODER_ID)
for p in vision_encoder.parameters(): p.requires_grad = False
vision_encoder.eval()
logger.info("Componentes carregados.")

## 6. Descongelamento Seletivo do Backbone

In [ ]:
# Descongelar os N blocos superiores do decoder
all_layers = list(student_backbone.model.layers)
n_total    = len(all_layers)
unfreeze_from = n_total - N_UNFREEZE_LAYERS

for i, layer in enumerate(all_layers):
    if i >= unfreeze_from:
        for p in layer.parameters():
            p.requires_grad = True
        logger.info("Bloco %d descongelado.", i)

# Descongelar também a cabeça de linguagem (lm_head)
for p in student_backbone.lm_head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in student_backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in student_backbone.parameters())
logger.info(
    "Student backbone: %s / %s parâmetros treináveis (%.1f%%)",
    f"{trainable:,}", f"{total:,}", 100 * trainable / max(total, 1),
)

In [ ]:
# Carregar conector da Fase 1
class ModalityConnector(nn.Module):
    def __init__(self, d_enc: int, d_model: int, d_hidden: Optional[int] = None) -> None:
        super().__init__()
        if d_hidden is None: d_hidden = max(d_enc, d_model)
        self.proj_in  = nn.Linear(d_enc, d_hidden, bias=True)
        self.act      = nn.GELU()
        self.proj_out = nn.Linear(d_hidden, d_model, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(next(self.parameters()).dtype)
        return self.proj_out(self.act(self.proj_in(x)))

connector = ModalityConnector(D_ENC_VISION, D_MODEL, CONNECTOR_HIDDEN).to(DTYPE_HIGH).to(DEVICE)

if PHASE1_CKPT.exists():
    ckpt = torch.load(PHASE1_CKPT, map_location=DEVICE)
    connector.load_state_dict(ckpt["model_state_dict"])
    logger.info("Pesos do conector (Fase 1) carregados de: %s", PHASE1_CKPT)
else:
    logger.warning("Checkpoint da Fase 1 não encontrado. Usando conector com pesos aleatórios.")

## 7. Definição da Loss Composta

In [ ]:
def composite_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    labels: torch.Tensor,
    student_embeds: torch.Tensor,
    teacher_embeds: torch.Tensor,
    w_language: float = W_LANGUAGE,
    w_distill: float  = W_DISTILL,
    w_align: float    = W_ALIGN,
    temperature: float = TEMP_DISTILL,
) -> Dict[str, torch.Tensor]:
    """
    Compute the composite training loss for Phase 2 multimodal alignment.

    Combines autoregressive language loss, KL distillation loss, and
    cosine representation alignment loss, as specified in Section 6
    of the architecture specification.

    Parameters
    ----------
    student_logits : torch.Tensor
        Student model output logits, shape (batch, seq_len, vocab_size).
    teacher_logits : torch.Tensor
        Teacher model output logits, same shape.
    labels : torch.Tensor
        Ground-truth token labels, shape (batch, seq_len).
    student_embeds : torch.Tensor
        Student connector output embeddings for alignment.
    teacher_embeds : torch.Tensor
        Teacher encoder output embeddings for alignment.
    w_language, w_distill, w_align : float
        Loss component weights.
    temperature : float
        Distillation softmax temperature.

    Returns
    -------
    dict of str to torch.Tensor
        Dictionary with keys 'language', 'distill', 'align', 'total'.
    """
    # Language loss
    sl = student_logits[:, :-1].contiguous()
    tgt = labels[:, 1:].contiguous()
    l_lang = F.cross_entropy(sl.view(-1, sl.size(-1)), tgt.view(-1), ignore_index=-100)

    # KL distillation loss
    s_log = F.log_softmax(student_logits / temperature, dim=-1)
    t_prob = F.softmax(teacher_logits.detach() / temperature, dim=-1)
    l_distill = F.kl_div(
        s_log.view(-1, s_log.size(-1)), t_prob.view(-1, t_prob.size(-1)),
        reduction="batchmean"
    ) * (temperature ** 2)

    # Representation alignment loss (cosine)
    se = student_embeds.view(-1, student_embeds.size(-1))
    te = teacher_embeds.view(-1, teacher_embeds.size(-1)).detach()
    l_align = 1.0 - F.cosine_similarity(se, te, dim=-1).mean()

    total = w_language * l_lang + w_distill * l_distill + w_align * l_align
    return {"language": l_lang, "distill": l_distill, "align": l_align, "total": total}

## 8. Preparação dos Dados

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

class MultimodalAlignmentDataset(Dataset):
    """
    Dataset for Phase 2 multimodal alignment training.

    Parameters
    ----------
    hf_dataset : datasets.Dataset
        Dataset containing 'image' and caption fields.
    vision_processor : CLIPProcessor
        Processor for the vision encoder.
    tokenizer : PreTrainedTokenizer
        Tokenizer for the language backbone.
    max_text_len : int
        Maximum caption token length.
    """

    def __init__(self, hf_dataset, vision_processor, tokenizer, max_text_len: int = 96):
        self.dataset = hf_dataset
        self.vp = vision_processor
        self.tok = tokenizer
        self.max_len = max_text_len

    def __len__(self) -> int: return len(self.dataset)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.dataset[idx]
        image = item["image"].convert("RGB")
        caption = item["captions"][0] if isinstance(item.get("captions"), list) else item.get("caption", "")
        pix = self.vp(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
        tok = self.tok(caption, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return {
            "pixel_values": pix,
            "input_ids": tok["input_ids"].squeeze(0),
            "attention_mask": tok["attention_mask"].squeeze(0),
        }

raw = load_dataset("nlphuji/flickr30k", split="test[:1000]")
ds  = MultimodalAlignmentDataset(raw, vision_processor, tokenizer)
dl  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
logger.info("Dataset pronto: %d amostras | %d batches/epoch", len(ds), len(dl))

## 9. Loop de Alinhamento Multimodal

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Grupos de parâmetros com learning rates diferenciadas
param_groups = [
    {"params": connector.parameters(),
     "lr": LR_CONNECTOR, "weight_decay": 1e-2},
    {"params": [p for p in student_backbone.parameters() if p.requires_grad],
     "lr": LR_BACKBONE, "weight_decay": 1e-2},
]
optimiser = AdamW(param_groups, betas=(0.9, 0.95), eps=1e-8)

total_steps = N_EPOCHS * len(dl)
scheduler = SequentialLR(
    optimiser,
    [LinearLR(optimiser, 1e-3, 1.0, WARMUP_STEPS),
     CosineAnnealingLR(optimiser, T_max=max(total_steps - WARMUP_STEPS, 1), eta_min=1e-6)],
    milestones=[WARMUP_STEPS],
)

phase2_metrics: List[Dict] = []
global_step: int = 0

for epoch in range(N_EPOCHS):
    student_backbone.train()
    connector.train()
    accum: Dict[str, float] = {k: 0.0 for k in ("total", "language", "distill", "align")}
    n_steps: int = 0

    for step, batch in enumerate(dl):
        pv      = batch["pixel_values"].to(DEVICE, dtype=DTYPE_HIGH)
        ids     = batch["input_ids"].to(DEVICE)
        mask    = batch["attention_mask"].to(DEVICE)

        with torch.no_grad():
            vis_feats    = vision_encoder(pixel_values=pv).last_hidden_state
            text_embeds  = student_backbone.get_input_embeddings()(ids)
            teacher_vis  = vision_encoder(pixel_values=pv).last_hidden_state

        projected = connector(vis_feats)
        combined  = torch.cat([projected, text_embeds], dim=1)
        comb_mask = torch.cat([
            torch.ones(projected.shape[:2], device=DEVICE, dtype=mask.dtype), mask
        ], dim=1)
        n_vis  = projected.shape[1]
        labels = torch.cat([
            torch.full((ids.shape[0], n_vis), -100, device=DEVICE), ids
        ], dim=1)

        # Student forward
        student_out = student_backbone(inputs_embeds=combined, attention_mask=comb_mask)

        # Teacher forward (no grad)
        with torch.no_grad():
            teacher_out = teacher(inputs_embeds=combined, attention_mask=comb_mask)

        losses = composite_loss(
            student_logits=student_out.logits,
            teacher_logits=teacher_out.logits,
            labels=labels,
            student_embeds=projected,
            teacher_embeds=teacher_vis,
        )

        optimiser.zero_grad()
        losses["total"].backward()
        nn.utils.clip_grad_norm_(
            list(connector.parameters()) +
            [p for p in student_backbone.parameters() if p.requires_grad],
            GRAD_CLIP,
        )
        optimiser.step()
        scheduler.step()

        for k in accum: accum[k] += losses[k].item()
        n_steps += 1; global_step += 1

        if (step + 1) % LOG_INTERVAL == 0:
            lr = scheduler.get_last_lr()[0]
            logger.info(
                "Epoch %d | Step %d | total=%.4f | lang=%.4f | distill=%.4f | align=%.4f | lr=%.2e",
                epoch+1, step+1,
                accum["total"]/n_steps, accum["language"]/n_steps,
                accum["distill"]/n_steps, accum["align"]/n_steps, lr,
            )

    ep_avg = {k: v / max(n_steps, 1) for k, v in accum.items()}
    ep_avg["epoch"] = epoch + 1
    phase2_metrics.append(ep_avg)
    logger.info("Epoch %d concluída. Total loss: %.4f", epoch+1, ep_avg["total"])

## 10. Métricas de Alinhamento

In [ ]:
# Calcular similaridade de cosseno média pós-treinamento (deve ser próxima de 1.0)
student_backbone.eval()
connector.eval()

sample = next(iter(DataLoader(ds, batch_size=8)))
with torch.no_grad():
    pv = sample["pixel_values"].to(DEVICE, dtype=DTYPE_HIGH)
    vis = vision_encoder(pixel_values=pv).last_hidden_state
    proj = connector(vis)

cos_sim = F.cosine_similarity(
    proj.view(-1, proj.size(-1)),
    vis.view(-1, vis.size(-1))[:proj.view(-1, proj.size(-1)).shape[0]],
    dim=-1
).mean().item()

logger.info("Similaridade cosseno (connector output vs. encoder output): %.4f", cos_sim)
print(f"\n  Similaridade de cosseno pós-alinhamento: {cos_sim:.4f}\n")

In [ ]:
# Salvar checkpoint do student backbone + conector
torch.save({
    "student_backbone_state": {
        k: v for k, v in student_backbone.state_dict().items()
        # Apenas parâmetros que foram treinados
    },
    "connector_state": connector.state_dict(),
    "optimiser_state": optimiser.state_dict(),
    "n_unfreeze_layers": N_UNFREEZE_LAYERS,
    "epoch": N_EPOCHS,
}, CHECKPOINT_DIR / "phase2_aligned.pt")

with open(METRICS_DIR / "phase2_metrics.json", "w") as f:
    json.dump({"alignment_loss_per_epoch": phase2_metrics, "final_cosine_sim": cos_sim}, f, indent=2)

logger.info("Artefatos da Fase 2 persistidos.")
print("Artefatos da Fase 2 salvos com sucesso.")

## 12. Conclusões e Próximos Passos

A Fase 2 completou o alinhamento multimodal utilizando distilação de conhecimento e alinhamento de representações. O backbone foi adaptado ao domínio multimodal através de descongelamento seletivo de `N_UNFREEZE_LAYERS` blocos superiores, mantendo os encoders em FP16/BF16 e a integridade das camadas inferiores.

### Próxima Fase

Prosseguir para `03_ternary_transition.ipynb` (**Fase 3**) que converte o backbone textual para regime ternário via QAT contínuo, conforme Seção 5.4 da especificação técnica.